# Chapter 9 — System Design
### ML Engineer (Production) Interview Playbook

**Topics:** Fraud Detection · Transaction Monitoring · ML Pipeline

> This chapter is where all eight previous chapters get stitched together. In a system design
> interview, the interviewer isn't looking for the "correct" answer — they're looking at your
> **thought process**: do you clarify requirements? Do you estimate scale? Do you see and explicitly
> state the trade-offs? The three designs in this chapter map directly onto a fintech's (e.g. bunq's)
> real business.

## 9.0 A Framework for Answering

Answer every system design question with these five steps. Never jump straight to drawing boxes:

| Step | What you do | Rough time |
|---|---|---|
| 1) Clarify requirements | Ask questions: functional and non-functional; narrow the scope | 5 min |
| 2) Estimate scale | Put numbers on QPS, data volume, target latency | 5 min |
| 3) High-level design | Main components and the data flow between them | 10 min |
| 4) Go deep | Open up one or two critical components (usually whichever the interviewer asks about) | 15 min |
| 5) Trade-offs | Bottlenecks, failure modes, and why this choice over that one | 5 min |

**Interview tip:** The most common mistake: jumping straight to architecture without asking questions.
The interviewer deliberately leaves the question ambiguous to see whether you recognize the ambiguity.
Your first sentence should be a clarifying question, not a solution.

### Functional vs. non-functional requirements

- **Functional:** what does the system do? (e.g. "score and allow/block every transaction")
- **Non-functional:** how well? Latency, throughput, availability, consistency, cost. These shape the
  architecture.

## 9.1 Designing a Fraud Detection System

### Step 1: clarify requirements

Questions to ask (and our assumed answers):

- Must the decision be synchronous (blocking the transaction) or can it be async? → Synchronous
- What's the output: a score or a decision? → An allow/block/challenge decision
- Cost of a false positive vs. a false negative? → False positive is very costly (a real user gets
  blocked)
- Do we need explainability? → Yes, a regulatory requirement in banking

Final requirements:

- **Functional:** real-time scoring of every transaction and an allow / block / challenge (step-up
  verification) decision.
- **Non-functional:** p99 latency under 100ms; high availability; minimized false positives; fast model
  updates as new patterns emerge; full auditability.

### Step 2: estimate scale

Putting numbers on the problem shows you think like an engineer. A sample estimate:

In [ ]:
# Assumption: 10 million users, 5 transactions/user/day
# -> 50M transactions/day / 86400 seconds ~= 600 TPS average
# -> peak (5x) ~= 3000 TPS
#
# 100ms latency budget:
#   feature fetch   ~20ms
#   inference       ~30ms
#   rules + I/O     ~20ms
#   margin          ~30ms


**Interview tip:** Explicitly break down the **latency budget**. This immediately shows you
understand that 100ms is the *whole path*, not just the model's time — and it's exactly what reveals
that feature fetching can't be a heavy query on the primary database.

### Step 3: high-level architecture

```
Transaction
  -> API Gateway (auth, rate limit, idempotency key)
  -> Fraud Service
       -> Feature Store (online: Redis)  <- precomputed features
       -> Model inference (a lightweight model)
       -> Rules engine  <- safety layer and regulatory rules
       -> Decision (allow / block / challenge)
  -> Kafka (audit log + training data)
       -> Offline: retraining, analysis, a heavier review model
```

Workflow: a transaction enters the gateway (auth, rate limit, idempotency key check). The fraud service
reads real-time features from the online store, combines them with the current transaction's features,
gets a score from the model, and passes the score through the rules engine. The decision returns, and
the event is published to Kafka asynchronously.

### Step 4: going deeper into components

**a) The feature layer — the main latency bottleneck**

Features fall into three categories, each with a different path:

| Feature type | Example | Where it comes from |
|---|---|---|
| Transaction feature | Amount, currency, destination country, card type | The request payload itself (free) |
| Aggregate (windowed) feature | Transaction count in the last hour, 30-day average amount | Precomputed in the online store (Redis) |
| Profile feature | Account age, user's historical risk level | Online store (updated periodically) |

**Note:** The key point that scores well: aggregate features can't be computed on the fly with a query
against the primary database (slow and heavy load). They must be continuously updated by a streaming
job (like Flink) and written to Redis, so inference time only needs one fast read.

**b) Model + rules engine**

Why both? The model captures complex patterns but is a black box and hasn't been trained on a
brand-new pattern. The rules engine is fast, transparent, and instantly changeable — if a new fraud
pattern is discovered today, a rule can block it before the model is retrained. Rules also enforce
strict regulatory requirements (hard blocks).

In [ ]:
# Execution order: hard rules first (fail fast), then the model

def evaluate(tx, features) -> Decision:
    if rules.hard_block(tx):          # e.g. a sanctioned country
        return Decision.BLOCK

    score = model.predict(features)   # a lightweight model, ~30ms

    if score > 0.95:
        return Decision.BLOCK
    if score > 0.70:
        return Decision.CHALLENGE     # step-up verification -- reduces false positives

    return Decision.ALLOW


**Interview tip:** Why challenge? This middle path is the key to reducing the cost of false
positives. Instead of a binary "block or allow" choice, in the gray zone we ask the user for step-up
verification: a real user simply confirms, and a fraudster is stopped. This is a product design
decision the interviewer likes to hear.

**c) Resilience — what happens when something breaks?**

- **If Redis (the feature store) goes down?** With a short timeout, fall back: decide based only on
  rules and the transaction's own features (graceful degradation). The system must not halt all
  payments.
- **If the model service is slow/down?** The circuit breaker opens and we fall back to a conservative
  default policy (e.g. challenge for high-value transactions, allow for low-value ones).
- **A duplicate request?** An idempotency key at the gateway guarantees a transaction isn't
  double-processed or double-deducted.

### Step 5: trade-offs

| Trade-off | The two ends | Our choice and why |
|---|---|---|
| Latency ↔ Accuracy | A heavier, more accurate model vs. a slow one | A lightweight model for the real-time path + a heavier async model for review and discovery |
| FP ↔ FN | Blocking more = less fraud but unhappy users | A three-tier threshold with a challenge zone; the threshold is tuned to business cost |
| Model ↔ Rules | A stronger model but slow to react | Both: rules for instant reaction and regulatory requirements, the model for complex patterns |
| Consistency ↔ Availability | CAP in a financial system | Lean toward consistency, but with controlled degradation instead of a full outage |

### Q9.1 — How do you guarantee p99 latency stays under 100ms?

First I break down the latency budget and see how much each component contributes. The biggest risk is
feature fetching: no aggregate feature should be computed on the fly; everything is precomputed by a
streaming job and written to the online store (Redis) so only a sub-millisecond read is needed. I keep
the model lightweight (e.g. gradient boosting instead of a heavy deep network) and load it at startup.
I set hard timeouts on every dependency so a slow one doesn't eat the whole budget — and if it times
out, I fall back to the rules path instead of waiting. For p99 (not just the average) I have to watch
for garbage collection, cold starts, and request queueing, so proper autoscaling and connection pooling
are needed. And finally I continuously monitor p95/p99, not the average, which hides tail-distribution
problems.

## 9.2 Designing Transaction Monitoring (AML)

State the fundamental difference from fraud detection right away — it shows you've understood the
problem correctly:

| Feature | Fraud Detection | Transaction Monitoring (AML) |
|---|---|---|
| Unit of analysis | A single transaction | A behavioral pattern over time |
| Timing | Real-time (blocking) | Near-real-time / batch (non-blocking) |
| Output | An allow/block decision | An alert for human review (the compliance team) |
| Primary driver | Financial loss | Legal and regulatory requirement |

### Requirements and scale

- Detecting money-laundering patterns: structuring (breaking amounts under a reporting threshold),
  mule-account behavior, sudden volume spikes.
- Long time windows (hour, day, 30 days) per user.
- Full auditability: every alert must be reconstructible and explainable (a regulatory requirement).
- Latency: minutes is acceptable — unlike fraud, which needs milliseconds.

### High-level architecture

```
Transactions -> Kafka (event stream)
  -> Stream Processor (Flink / Kafka Streams)
       - time windows (tumbling / sliding / session)
       - stateful aggregation per user
       - pattern matching and AML rules
  -> Alert Service -> Compliance team review queue
  -> Data Warehouse (audit, regulatory reporting, analysis)
```

### Going deeper: stateful stream processing

This is the technical core of the design. For aggregation over time windows, the processor must keep
**state** — e.g. the sum of each user's transactions over the last 24 hours. This state can grow to
terabytes and must be resilient to failure.

- **Checkpointing:** the processor (e.g. Flink) periodically snapshots its state. If a node crashes, it
  recovers from the last checkpoint, not from scratch.
- **Window types:** tumbling (fixed, non-overlapping intervals), sliding (moving, with overlap), session
  (based on user activity). For structuring detection, a sliding window is appropriate.
- **Event time vs. processing time:** messages can arrive late or out of order. You must window based
  on "when the event happened," not "when it arrived," and use a watermark to decide how long to wait
  for late events.

**Interview tip:** Event time vs. processing time is one of the deepest points you can raise. If you
window based on processing time, a network delay can drop transactions into the wrong window and you
lose the structuring pattern. The watermark is the mechanism that controls this trade-off between
completeness and latency.

### The challenge: a high false-positive rate

Traditional rule-based AML systems have a very high false-positive rate (sometimes above 90% in the
industry), overwhelming the compliance team. The modern approach: layering.

- Layer 1 — Rules: covers regulatory requirements, transparent and defensible.
- Layer 2 — ML model: scores and prioritizes alerts so the most important ones get reviewed first.
- Layer 3 — Feedback: the outcome of human review flows back to the model and improves it.

**Note:** Why can't rules be fully replaced by ML? Because in a regulatory domain, you must be able to
explain to the regulator why an alert was generated. A black-box model isn't accountable. Common
pattern: rules for coverage and defensibility, ML for prioritization and noise reduction.

### Q9.2 — Why use stream processing for AML instead of periodic queries against a database?

Because AML patterns are based on per-user aggregation over time windows, and repeatedly running heavy
queries against the operational database is both slow and puts load on the primary payment system. A
stream processor keeps aggregate state incrementally: with every new transaction, counters and sums are
updated instead of rescanning the entire history each time. This is both far more efficient and drops
detection latency from hours to seconds/minutes. The important point is that this state must be
resilient to failure — via checkpointing — otherwise a node crash means losing in-progress windows. I
also need to window based on event time rather than processing time, so late or out-of-order events
don't corrupt the pattern.

## 9.3 Designing an ML Pipeline

A complete pipeline automates the end-to-end cycle from raw data to a production model. Goal:
reproducibility, automation, and safety.

```
Raw Data
  -> Validation (schema, quality, distribution)
  -> Feature Eng -> Feature Store (offline + online)
  -> Training (versioned data + code + config)
  -> Evaluation (holdout, comparison against baseline)
  --(quality gate)--> Model Registry
  -> Deploy (shadow -> canary -> full)
  -> Monitor (drift, quality, latency)
  --(trigger)--> Retrain
```

### Going deeper into key stages

**a) Validation — the first gate**

Bad data makes a bad model ("garbage in, garbage out"). This stage must stop the pipeline when there's
a problem, not silently pass it through:

- Schema checks (expected columns and types)
- Quality checks (NULL rate, out-of-range values, duplicate records)
- Distribution checks (is today's distribution drastically different from history? → a sign of an
  upstream failure)
- Tools: Great Expectations, pandera

**b) Quality Gate — the most important safeguard**

A new model is only deployed if it passes the quality gates. This is exactly what makes automated
retraining safe:

In [ ]:
# Quality gate before registering in the model registry

def quality_gate(new_model, baseline, holdout) -> bool:
    metrics = evaluate(new_model, holdout)

    if metrics.auc < baseline.auc - 0.01:        # a drop versus the current model
        return False
    if metrics.false_positive_rate > MAX_FPR:    # a business constraint
        return False
    if not passes_behavioral_tests(new_model):   # behavioral tests (Chapter 6)
        return False

    return True


**Note:** Automated retraining without a quality gate is the fastest way to ship a bad model to
production. Always say: "the new model must be compared against the current one and only deployed if
it's better (or at least not worse)" — and even then, gradually via shadow/canary.

**c) Idempotency and re-execution**

Every stage must be idempotent: re-running a stage with the same input must give the same output with
no doubled effect. Why? Because in a distributed system, stages fail and must be re-runnable without
corrupting data. A common pattern: write each stage's output under a deterministic key (e.g. date + code
version) so re-running simply overwrites.

### Orchestration

| Tool | Strength |
|---|---|
| Airflow | Scheduling batch DAGs; mature and widely used |
| Kubeflow Pipelines | Kubernetes-native; suited to heavy ML workloads |
| Metaflow / Prefect | Smoother developer experience, Python-first |

**Interview tip:** An important principle to state: every stage must be idempotent, re-runnable, and
versioned. The pipeline should be able to resume from the point of failure, not re-run the entire
multi-hour process from scratch.

### Q9.3 — How do you make automated retraining safe?

With several protective layers. First, data validation at the pipeline's input: if the training data
doesn't meet the required schema or quality, the pipeline stops and alerts — because retraining on bad
data is worse than not retraining. Second, a quality gate after training: the new model is evaluated on
a holdout and compared against the current (baseline) model; if it shows a metric drop or violates a
business constraint (like a false-positive-rate ceiling), it isn't deployed. Third, behavioral model
tests (invariance and directional) to make sure the base logic hasn't broken. Fourth, gradual
deployment: even an approved model goes to shadow first, then canary, not 100% of traffic. And finally,
the ability to roll back instantly to the previous version via the model registry. A subtle point: I
also need to watch for the feedback loop — if new training data comes only from transactions the current
model allowed, bias gets reinforced; that's why I keep a small control group.

## Quick Chapter Summary (Cheat Sheet)

Review this on interview day:

- **Framework:** 1) requirements (functional + non-functional) 2) scale estimate and latency budget
  3) high-level design 4) go deep 5) trade-offs. Never start without asking questions.
- **Fraud:** real-time <100ms; aggregate features precomputed in Redis (not on-the-fly queries); a
  lightweight model + rules engine; a challenge zone to reduce FP; fallback to rules on failure; Kafka
  for audit and retraining.
- **Monitoring/AML:** a behavioral pattern over time, not a single transaction; stream processing with
  state and checkpointing; event time + watermark; output is an alert for a human; rules for
  defensibility + ML for prioritization.
- **ML Pipeline:** validation → feature store → training (versioned) → quality gate → registry →
  shadow/canary → monitor → retraining trigger; every stage idempotent and re-runnable.
- **Golden trade-offs:** latency ↔ accuracy (lightweight real-time + heavy async); FP ↔ FN (threshold
  based on business cost); model ↔ rules (both, not either); C ↔ A (financial = lean consistency with
  controlled degradation).